# 3.1 Tickers
The goal is to solve the problems of the Polygon ticker lists in the introduction. Before we do that we will download the ticker list for all days from Polygon and store them into the map <code>tickers</code>.

In [1]:
from massive.rest import RESTClient
from datetime import datetime, date, time, timedelta
from pytz import timezone
from functools import lru_cache
from times import get_market_dates, first_trading_date_after, last_trading_date_before, first_trading_date_after_equal, last_trading_date_before_equal
import os
import pytz
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import mplfinance as mpf
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import glob
from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

client = RESTClient(api_key=API_KEY)

POLYGON_DATA_PATH = "../data/polygon/"

START_DATE = date(2016, 5, 24)
END_DATE = date(2023, 5, 21)

First, I will create a function to download the tickers for a specific date. 

We also need to download the 'None' type tickers:
- Incorrectly labeled stocks. 
    - 2856 stocks labeled as None/NaN rather than 'CS'!!! (even Mega-caps like AAPL and liquid stocks like AAOI).

In [2]:
def download_tickers(day):
    """Retrieve the ticker list for a specific date

    Args:
        day (Date): the Date for which to download the ticker list

    Returns:
        DataFrame: the ticker list
    """
    
    date_iso = day.isoformat()

    tickers_iterator_active = client.list_tickers(date=date_iso, active=True, market='stocks', limit=1000)
    tickers = pd.DataFrame(tickers_iterator_active)
    tickers = tickers[~tickers['type'].isin(['PFD', 'WARRANT', 'RIGHT', 'BOND', 'ETF', 'ETN', \
        'ETV', 'SP', 'ADRP', 'ADRW', 'ADRR', 'FUND', 'BASKET', 'UNIT', 'LT', 'GDR', 'OTHER', \
            'AGEN', 'EQLK', 'ETS', 'INDEX'])] # Only keep CS, ADRC, NYRS and OS (including None)

    tickers.sort_values(by = "ticker", inplace=True)
    tickers.reset_index(inplace=True, drop=True)
    return tickers[['ticker', 'name', 'active', 'delisted_utc', 'last_updated_utc', 'cik', 'composite_figi', 'type']]

Then all ticker lists are downloaded and stored in the <code>test/tickers/</code> map. But only the ones that we need if we already have some.

In [3]:
""" # Get a list of what we already have
files = os.listdir(POLYGON_DATA_PATH + 'test/tickers')
available_dates = [date.fromisoformat(file.replace(".csv", "")) for file in files if file.endswith(".csv")]

trading_dates = get_market_dates()
for day in trading_dates:
    # Only download what we do not have
    if day >= START_DATE and day <= END_DATE and day not in available_dates:
        tickers = download_tickers(day)
        tickers.to_csv(POLYGON_DATA_PATH + f"test/tickers/{day.isoformat()}.csv") """

' # Get a list of what we already have\nfiles = os.listdir(POLYGON_DATA_PATH + \'test/tickers\')\navailable_dates = [date.fromisoformat(file.replace(".csv", "")) for file in files if file.endswith(".csv")]\n\ntrading_dates = get_market_dates()\nfor day in trading_dates:\n    # Only download what we do not have\n    if day >= START_DATE and day <= END_DATE and day not in available_dates:\n        tickers = download_tickers(day)\n        tickers.to_csv(POLYGON_DATA_PATH + f"test/tickers/{day.isoformat()}.csv") '

A random ticker list:

In [3]:
pd.read_csv(POLYGON_DATA_PATH + f"test/tickers/2022-06-09.csv", index_col=0).head(3)

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
0,A,Agilent Technologies Inc.,True,NaN,2024-12-03T22:25:34.943853Z,1090872.0,BBG000C2V3D6,CS
1,AA,Alcoa Corporation,True,NaN,2024-12-03T22:25:34.944676Z,1675149.0,BBG00B3T3HD3,CS
2,AAC,Ares Acquisition Corporation,True,NaN,2024-12-03T22:25:34.943993Z,1829432.0,NaN,CS


We observe that the <code>last_updated_utc</code> does not match the date of the ticker list. For example for "A", this date is *after* 2022-06-09. So this value is not point-in-time. Nevertheless, this value is useless for us because we only use the data to determine the delisting date. Neither do we need <code>delisted_utc</code> for the same reason.

Later when we do have data, we will create a new column <code>start_data</code> and <code>end_data</code> which gives the start and end dates from the available data.

In [4]:
# just an intial number. As we see later, these are NOT unique at all (META vs META ETF).
unique_tickers = set()

files = os.listdir(POLYGON_DATA_PATH + 'test/tickers')

for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test/tickers/{file}', index_col=0)
    # Add unique tickers from this file to the set
    unique_tickers.update(df['ticker'].dropna().unique())

print(f"Total unique tickers across all files: {len(unique_tickers)}")


Total unique tickers across all files: 11536


## VFS ticker exists! 

In [5]:
# List to hold filtered DataFrames
vfs_frames = []

files = os.listdir(POLYGON_DATA_PATH + 'test/tickers')

for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test/tickers/{file}', index_col=0)
    
    # Filter rows where ticker == 'VFS'
    filtered = df[df['ticker'] == 'VFS']
    
    # Append if not empty
    if not filtered.empty:
        vfs_frames.append(filtered)

# Combine all filtered data into one DataFrame
if vfs_frames:
    vfs_df = pd.concat(vfs_frames, ignore_index=True)
else:
    vfs_df = pd.DataFrame()  # Empty DataFrame if no matches

print(f"Total rows with ticker 'VFS': {len(vfs_df)}")


Total rows with ticker 'VFS': 172


In [ ]:
# VFS "list_date": "2023-08-15" 
# https://massive.com/docs/rest/stocks/tickers/ticker-overview

vfs_df.sort_values(['last_updated_utc'], inplace=True)

vfs_df

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
106,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:05:57.474722Z,1913510.0,NaN,CS
97,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:05:58.868482Z,1913510.0,NaN,CS
57,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:05:59.171882Z,1913510.0,NaN,CS
80,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:06:01.646507Z,1913510.0,NaN,CS
73,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:06:02.251002Z,1913510.0,NaN,CS
...,...,...,...,...,...,...,...,...
168,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:43:34.208948Z,1913510.0,NaN,CS
110,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:44:32.053137Z,1913510.0,NaN,CS
119,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:44:47.549244Z,1913510.0,NaN,CS
24,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:45:43.157237Z,1913510.0,NaN,CS


# VFS remains an Active ticker so it should have come up later! Why didnt it?

In [7]:
vfs_df[vfs_df['active'] == False]

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type


# More problems

### The amount of None values for the ticker types is problematic. Sometimes, there isn't even a ticker description or even a ticker (what!). We will clean the ticker lists in the following way:
* Remove tickers with no description or no ticker (What!?)
* Assign ticker type 'NONE'. When merging in step 3.5, if the correct ticker type shows up we will use that. Else we will 'manually' take care of it, e.g. by assigning ADRC to all tickers that have 'DEP', 'ADR' and 'ADS' in its name.

**After around 2019, this issue was fixed. So we don't have to do this for new ticker lists.**
<!-- * Assign ticker type 'ADR' if the ticker description contains:
    * DEP, ADR, ADS, PLC, n.v., has a suffix with () that contains a non-US country. 
* Assign ticker type 'CS' otherwise.
Btw, there are also a lot of preferred stocks and other stuff.  -->

In [ ]:
total_no_name = 0
total_no_ticker = 0

missing_ticker_rows = []   # rows where ticker is NaN
missing_name_rows = []     # rows where name is NaN

files = os.listdir(POLYGON_DATA_PATH + 'test/tickers')

for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test/tickers/{file}', index_col=0)

    # Missing ticker
    mask_no_ticker = df['ticker'].isna()
    count_ticker = mask_no_ticker.sum()
    total_no_ticker += count_ticker
    if count_ticker > 0:
        tmp = df[mask_no_ticker].copy()
        tmp['source_file'] = file
        missing_ticker_rows.append(tmp)

    # Missing name
    mask_no_name = df['name'].isna()
    count_name = mask_no_name.sum()
    total_no_name += count_name
    if count_name > 0:
        tmp = df[mask_no_name].copy()
        tmp['source_file'] = file
        missing_name_rows.append(tmp)

# Save CSV for missing ticker
if missing_ticker_rows:
    df_missing_ticker = pd.concat(missing_ticker_rows, ignore_index=True)
    df_missing_ticker.to_csv("../data/polygon/tmp/" + 'missing_ticker_rows.csv', index=False)
    print(f"Saved {len(df_missing_ticker)} rows with missing ticker.")
else:
    df_missing_ticker = pd.DataFrame()
    print("No missing ticker rows.")

# Save CSV for missing name
if missing_name_rows:
    df_missing_name = pd.concat(missing_name_rows, ignore_index=True)
    df_missing_name.to_csv("../data/polygon/tmp/" + 'missing_name_rows.csv', index=False)
    print(f"Saved {len(df_missing_name)} rows with missing name.")
else:
    df_missing_name = pd.DataFrame()
    print("No missing name rows.")

# Totals (includes lots of duplicates - only very few are unique)
print(f"Number of tickers with missing ticker: {total_no_ticker}")
print(f"\nNumber of tickers with no name: {total_no_name}")

# Unique tickers in the missing‑name rows (your request)
if not df_missing_name.empty:
    # dropna() because ticker could also be NaN in these rows
    unique_tickers_missing_name = df_missing_name['ticker'].dropna().unique()
    print(f"\nUnique tickers among rows with missing name: {len(unique_tickers_missing_name)}")
    # optional: show first 50
    print(unique_tickers_missing_name[:50])
else:
    print("\nNo missing name rows, so no unique tickers to show.")

Really only 1 company is missing it's ticker (for "Nano Labs Ltd American Depositary Shares", "Nano Labs Ltd Class A Ordinary Shares")

Really only 20 tickers are missing names. And never heard of these tickers anyways.

In [11]:
files = os.listdir(POLYGON_DATA_PATH + 'test/tickers')
for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test/tickers/{file}', index_col=0)
    df = df[~df['name'].isna()] # Remove tickers with no name
    df = df[~df['ticker'].isna()]
    df['type'] = df['type'].fillna('NONE')
    # df.reset_index(drop=True).to_csv(POLYGON_DATA_PATH + f'test/tickers/{file}')
    df.reset_index(drop=True).to_csv(POLYGON_DATA_PATH + f'test_processed/tickers/{file}')

In [12]:
total_no_name = 0
total_no_ticker = 0

missing_ticker_rows = []   # rows where ticker is NaN
missing_name_rows = []     # rows where name is NaN

files = os.listdir(POLYGON_DATA_PATH + 'test_processed/tickers')

for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test_processed/tickers/{file}', index_col=0)

    # Missing ticker
    mask_no_ticker = df['ticker'].isna()
    count_ticker = mask_no_ticker.sum()
    total_no_ticker += count_ticker
    if count_ticker > 0:
        tmp = df[mask_no_ticker].copy()
        tmp['source_file'] = file
        missing_ticker_rows.append(tmp)

    # Missing name
    mask_no_name = df['name'].isna()
    count_name = mask_no_name.sum()
    total_no_name += count_name
    if count_name > 0:
        tmp = df[mask_no_name].copy()
        tmp['source_file'] = file
        missing_name_rows.append(tmp)

# Save CSV for missing ticker
if missing_ticker_rows:
    df_missing_ticker = pd.concat(missing_ticker_rows, ignore_index=True)
    # df_missing_ticker.to_csv("../data/polygon/tmp/" + 'test_processed_missing_ticker_rows.csv', index=False)
    print(f"Saved {len(df_missing_ticker)} rows with missing ticker.")
else:
    df_missing_ticker = pd.DataFrame()
    print("No missing ticker rows.")

# Save CSV for missing name
if missing_name_rows:
    df_missing_name = pd.concat(missing_name_rows, ignore_index=True)
    # df_missing_name.to_csv("../data/polygon/tmp/" + 'test_processed_missing_name_rows.csv', index=False)
    print(f"Saved {len(df_missing_name)} rows with missing name.")
else:
    df_missing_name = pd.DataFrame()
    print("No missing name rows.")

# Totals (includes lots of duplicates - only very few are unique)
print(f"Number of tickers with missing ticker: {total_no_ticker}")
print(f"\nNumber of tickers with no name: {total_no_name}")

# Unique tickers in the missing‑name rows (your request)
if not df_missing_name.empty:
    # dropna() because ticker could also be NaN in these rows
    unique_tickers_missing_name = df_missing_name['ticker'].dropna().unique()
    print(f"\nUnique tickers among rows with missing name: {len(unique_tickers_missing_name)}")
    # optional: show first 50
    print(unique_tickers_missing_name[:50])
else:
    print("\nNo missing name rows, so no unique tickers to show.")

No missing ticker rows.
No missing name rows.
Number of tickers with missing ticker: 0

Number of tickers with no name: 0

No missing name rows, so no unique tickers to show.


In [13]:
unique_tickers = set()

files = os.listdir(POLYGON_DATA_PATH + 'test_processed/tickers')

for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test_processed/tickers/{file}', index_col=0)
    # Add unique tickers from this file to the set
    unique_tickers.update(df['ticker'].dropna().unique())

print(f"Total unique tickers across all files: {len(unique_tickers)}")

Total unique tickers across all files: 11514


Btw, some ADRs are classified as common stocks...

# 3.2 Building the tickers loop
Now we can finally create our ticker list, which includes all tickers. The process involves looping over all Polygon ticker lists and updating our own one. First some notation: T is our ticker list that we iteratively update using Polygons ticker list. P(i) is the Polygon ticker list from day *i*. 

1. On day 1, our ticker list is the same as the one from Polygon, but with some extra columns. We create a column <code>start_date</code> which is day 1 and <code>end_date</code> with is empty. We are only interested in stocks that were active on that day.
2. For all *i = 2 ... n* days, for the active stocks:
    * **Delistings**: The stocks that are in T but not in P(i) are the stocks that are removed by Polygon (e.g. FB). For these tickers we set the <code>end_date</code> in T to day *i-1*. 
    * **New listings**: The stocks that are in P(i) but not in T are the new listings. We will append the new stock to T and set the start_date to day *i*.
    * **Everything else**: The stocks that are both in P(i) and T are the stocks that 'continue their listings'. We do nothing.

Two tickers are the 'same' if all fields except <code>last_updated_utc</code> or <code>delisted_utc</code> are the same.

For testing, we will start with 2022-06-08 and update to 2022-06-09. Both FB and META should then be included with the correct start and end dates. The start and end date of FB should be 2022-06-08 and the start date of META should be 2022-06-09. The end date of META should be empty.

"The stock ticker for Meta Platforms, Inc. officially changed from FB to META on June 9, 2022 at the start of the trading day / Open".

Massive confirms this too! https://massive.com/docs/rest/stocks/corporate-actions/ticker-events (Experimental Endpoint though! So don't go by this yet!)

SO WHY IS DATA DIFFERENT! AND WHY DIFFERENT NOW AND NOT A YEAR AGO!

Example of issue:

META mega-cap started trading on June 9, 2022 at the start of the trading day / Open. OHLCV data only exists for META from 2022-06-09

FB mega-cap stopped trading on June 8, 2022. OHLCV data only exists for FB till 2022-06-08.

https://massive.com/docs/rest/stocks/aggregates/custom-bars endpoint confirms that OHLCV data only exists for META from 2022-06-09, and only exists for FB till 2022-06-08.

So this confirms that Massive changed things and now shows the Old ticker for 1 more day even when it's the New ticker trading on that date and not the old! 
(maybe is now using that Ticker Change experimental endpoint that shows 2022-06-09 as Ticker change date)

And if Massive is doing this for a Mega-cap like META, it's doing it for ALL stocks, so we can be rest assured that we can make this change. 
Besides, it's better to make this change a day earlier than the Event date. 

So we need to use day1-1 for the end_date.

**Basically, the OHLCV endpoint gives no data for FB and should be the one we need to be using to check End-date!**



In [8]:
day_1 = date(2022, 6, 8)
day_2 = date(2022, 6, 9)

our_tickers = pd.read_csv(
    POLYGON_DATA_PATH + f"test/tickers/{day_1.isoformat()}.csv",
    index_col=0,
)
our_tickers = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]]
our_tickers = our_tickers[our_tickers["active"] == True]
our_tickers.reset_index(inplace=True, drop=True)

our_tickers["start_date"] = day_1.isoformat()
our_tickers["end_date"] = pd.NaT

tickers_day_2 = pd.read_csv(
    POLYGON_DATA_PATH + f"test/tickers/{day_2.isoformat()}.csv",
    index_col=0,
)
tickers_day_2 = tickers_day_2[["ticker", "name", "active", "cik", "composite_figi", "type"]]
tickers_day_2 = tickers_day_2[tickers_day_2["active"] == True]
tickers_day_2.reset_index(inplace=True, drop=True)

In [9]:
our_tickers.head(2)

,ticker,name,active,cik,composite_figi,type,start_date,end_date
0,A,Agilent Technologies Inc.,True,1090872.0,BBG000C2V3D6,CS,2022-06-08,NaT
1,AA,Alcoa Corporation,True,1675149.0,BBG00B3T3HD3,CS,2022-06-08,NaT


In [10]:
tickers_day_2.head(2)

,ticker,name,active,cik,composite_figi,type
0,A,Agilent Technologies Inc.,True,1090872.0,BBG000C2V3D6,CS
1,AA,Alcoa Corporation,True,1675149.0,BBG00B3T3HD3,CS


Preliminary check for duplicates

In [11]:
# Preliminary check: no duplicates
if our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]].duplicated().any():
    raise Exception("There are duplicates!")

if tickers_day_2[["ticker", "name", "active", "cik", "composite_figi", "type"]].duplicated().any():
    raise Exception("There are duplicates!")

We will first get the delisting and new listings. (Nothing has to be done with the kept listings).

In [12]:
# DELISTINGS: Get tickers that are in T but not in P(2). This is actually not straightforward 
# (https://stackoverflow.com/questions/28901683/pandas-get-rows-which-are-not-in-other-dataframe). 
# We need to get the rows in tickers_day_2 that are not in our_tickers. 
# We will use the merge function but specifying indicator=True and use a left merge (tickers_day_2 left merge to our_tickers). 
# What gets returned is a dataframe with the flags "left_only", "right_only" and "both". 
# If the indicator is "left_only", it means that it existed in only in the left DataFrame (our_tickers). This is exactly what we need. 
indicator_delisted = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]].merge(tickers_day_2[["ticker", "name", "active", "cik", "composite_figi", "type"]], on=["ticker", "name", "active", "cik", "composite_figi", "type"], 
                   how='left', indicator=True)
indicator_delisted = indicator_delisted["_merge"] # Only get the indicator

delisted_tickers = our_tickers[indicator_delisted == "left_only"] # Only get the delisted tickers

# NEW LISTINGS: Swap the DataFrames
indicator_new = tickers_day_2[["ticker", "name", "active", "cik", "composite_figi", "type"]].merge(our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]], on=["ticker", "name", "active", "cik", "composite_figi", "type"], 
                   how='left', indicator=True)
indicator_new = indicator_new["_merge"]
new_tickers = tickers_day_2[indicator_new == "left_only"]

# KEPT LISTINGS
current_tickers = our_tickers[indicator_delisted == "both"] # “both” if found in both DataFrames.
# current_tickers = tickers_day_2[indicator_new == "both"] # It does not matter which one we choose

In [13]:
print(len(delisted_tickers))
delisted_tickers.head(2)

4


,ticker,name,active,cik,composite_figi,type,start_date,end_date
1181,CERN,Cerner Corp,True,804753.0,BBG000BFDLV8,CS,2022-06-08,NaT
1851,DYNS,Dynamics Special Purpose Corp. Class A Common ...,True,1854270.0,BBG010WX7ZB3,CS,2022-06-08,NaT


In [14]:
print(len(new_tickers))
new_tickers.head(2)

5


,ticker,name,active,cik,composite_figi,type
2508,GLAQ,Globis Acquisition Corp. common stock,True,1823383.0,NaN,CS
2806,HOUS,Anywhere Real Estate Inc.,True,1398987.0,BBG000QN4GY3,CS


In [15]:
len(current_tickers)

6452

Then we will process the delistings and listings.

In [16]:
# DELISTINGS: register delisting date and set to inactive.
# our_tickers.loc[indicator_delisted == "left_only", "end_date"] = day_1 #Not day_2!
# our_tickers.loc[indicator_delisted == "left_only", "end_date"] = day_1.isoformat() #Not day_2!
our_tickers.loc[indicator_delisted == "left_only", "end_date"] = day_1.isoformat() #Not day_2!
our_tickers.loc[indicator_delisted == "left_only", "active"] = False

our_tickers[our_tickers["ticker"] == "FB"]

,ticker,name,active,cik,composite_figi,type,start_date,end_date
2141,FB,"Meta Platforms, Inc. Class A Common Stock",True,1326801.0,BBG000MM2P62,CS,2022-06-08,NaT


In [17]:
# NEW LISTINGS: append the new tickers and register start date
print(len(our_tickers))
print(len(new_tickers))

our_tickers = pd.concat([our_tickers, new_tickers])
our_tickers.reset_index(inplace=True, drop=True)
our_tickers['start_date'] = our_tickers['start_date'].fillna(value=day_2)

print(len(our_tickers))
our_tickers[our_tickers["ticker"] == "META"]

6456
5
6461


,ticker,name,active,cik,composite_figi,type,start_date,end_date
6458,META,"Meta Platforms, Inc. Class A Common Stock",True,1326801.0,BBG000MM2P62,CS,2022-06-09,NaT


Some final checks and setting <code>end_date</code> for the active listings at END_DATE.

In [18]:
# Errors cos of missing values in "type" column
""" if our_tickers[["ticker", "name", "active", "type", "start_date"]].isnull().values.any():
    raise Exception("There are missing values.")

# After all is done, set the end_date for active stocks to the new day. This is only done after all iterations. 
our_tickers["end_date"].fillna(value=day_2, inplace=True) """

# Removed "type" from the required list of columns with no missing values. Cos we surmized earlier when looking for 'CS' types that some tickers are missing values for "type".
# required_cols = ["ticker", "name", "active", "type", "start_date"]
required_cols = ["ticker", "name", "active", "start_date"]


# Find missing rows
missing_rows = our_tickers[our_tickers[required_cols].isnull().any(axis=1)]

if not missing_rows.empty:
    print("Missing values found in the following rows:")
    print(missing_rows)

    # Get columns with missing values (only from required_cols)
    cols_with_missing = [col for col in required_cols if our_tickers[col].isnull().any()]
    print(f"\nColumns containing missing values: {cols_with_missing}")

    raise Exception(f"There are missing values in columns: {cols_with_missing}\nMissing rows:\n{missing_rows.to_string()}")

The result is correct. FB is included with the correct <code>end_date</code>. Then META starts with the correct <code>start_date</code>.

In [19]:
our_tickers[our_tickers['ticker'].isin(['FB', 'META'])]

,ticker,name,active,cik,composite_figi,type,start_date,end_date
2141,FB,"Meta Platforms, Inc. Class A Common Stock",True,1326801.0,BBG000MM2P62,CS,2022-06-08,NaT
6458,META,"Meta Platforms, Inc. Class A Common Stock",True,1326801.0,BBG000MM2P62,CS,2022-06-09,NaT


Further down the line, I got an error that I wouldn't have gotten if I checked for duplicates among the active tickers given a day. For the raw polygon ticker lists, I assumed that for all active tickers on that day, there were no duplicates. However I was wrong:

In [26]:
our_tickers = pd.read_csv(
    POLYGON_DATA_PATH + f"test/tickers/2024-03-01.csv",
    index_col=0,
)
our_tickers = our_tickers[our_tickers["active"] == True]
our_tickers.sort_values(['last_updated_utc'])
our_tickers.reset_index(inplace=True, drop=True)

duplicated = our_tickers[our_tickers["ticker"].duplicated(keep=False)]
duplicated[duplicated["ticker"].duplicated(keep=False)]

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type


We need to make sure to remove these. We will simply keep the last one as that is the most recent one (we have already sorted on 'last_updated_utc').

In [27]:
""" indices_to_remove = duplicated["ticker"].duplicated(keep='last')
list(indices_to_remove[indices_to_remove].index) """

' indices_to_remove = duplicated["ticker"].duplicated(keep=\'last\')\nlist(indices_to_remove[indices_to_remove].index) '

In [28]:
""" our_tickers.drop(list(indices_to_remove[indices_to_remove].index), inplace=True)
our_tickers.reset_index(drop=True, inplace=True) """

' our_tickers.drop(list(indices_to_remove[indices_to_remove].index), inplace=True)\nour_tickers.reset_index(drop=True, inplace=True) '

# No longer an issue. But let's keep the "cleaning" code so our data is future-proof.

I also found another type of duplicate:

In [29]:
our_tickers = pd.read_csv(
    POLYGON_DATA_PATH + f"test/tickers/2016-07-01.csv",
    index_col=0,
)
our_tickers[our_tickers["ticker"] == 'CMCSA']

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type


In [30]:
our_tickers[our_tickers["ticker"] == 'CMCS.A']

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
1111,CMCS.A,COMCAST CORP CL A (NEW),True,NaN,2025-01-15T22:33:44.852864Z,1166691.0,BBG000BFT2L4,NaN


We will also remove these.

In [31]:
for day in get_market_dates():
    our_tickers = pd.read_csv(POLYGON_DATA_PATH + f'test/tickers/{day.isoformat()}.csv', index_col=0)
    indices_to_remove = []
    our_tickers_with_points = our_tickers[our_tickers['ticker'].str.contains(r'\.')]
    
    for index, row in our_tickers_with_points.iterrows():
        ticker = row['ticker']
        name = row['name']
        if '.' in ticker:
            ticker_without_point = ticker.replace('.', '')
            duplicate = our_tickers[our_tickers['ticker'] == ticker_without_point]
            if duplicate.size > 0:
                if duplicate['name'].values[0] == name:
                    indices_to_remove.append(duplicate.index[0])
    
    # --- VERIFICATION SECTION ---
    if indices_to_remove:
        print(f"Date: {day.isoformat()} -> Number of rows to remove: {len(indices_to_remove)}")
        print("Rows that will be removed (non‑dotted tickers with matching name):")
        print(our_tickers.loc[indices_to_remove].to_string())
        print("-" * 50)
    # ----------------------------

print(indices_to_remove)
print(len(indices_to_remove))

[]
0


In [32]:
if len(indices_to_remove) > 0:
    print("removing duplicates")
    our_tickers.drop(indices_to_remove, inplace=True)
    our_tickers.reset_index(drop=True, inplace=True)
    our_tickers.to_csv(POLYGON_DATA_PATH + f'test/tickers/{day.isoformat()}.csv')
else:
    print("Nothing to do")

Nothing to do


# Yup, no name mismatches and hence, no duplicate to remove! Massive have cleaned this bit! But keeping the code above as future-proofing.

# 3.3 The tickers loop
The ticker lists of 2009-10-29, 2010-03-30 and 2010-03-31 seem incomplete. We will simply copy the ticker list the trading day before to avoid errors.

Putting it all in a loop gives the following code. We save the results to <code>tickers_v1.csv</code>.

In [ ]:
# --- CONVERT DATE RANGE TO PANDAS TIMESTAMP ---
# Changed: Use pd.Timestamp instead of date to avoid dtype issues
START_DATE = pd.Timestamp(date(2016, 5, 24))
END_DATE = pd.Timestamp(date(2023, 5, 24))

# Assumed existing functions (they must return date-like objects)
# We'll convert their results to Timestamp as well
market_dates_raw = get_market_dates()  # list of datetime.date or Timestamp
market_days = [pd.Timestamp(d) for d in market_dates_raw]  # force Timestamp

# Adjust to trading days – convert to date for comparison, then back to Timestamp
START_DATE = pd.Timestamp(first_trading_date_after_equal(START_DATE.date()))
END_DATE = pd.Timestamp(last_trading_date_before_equal(END_DATE.date()))

print(START_DATE)
print(END_DATE)
print(market_days[:5], market_days[-5:])

# Initialize our_tickers as None
our_tickers = None

for day in market_days:
    if day == START_DATE:
        # At the start, our ticker list is the same as polygon.
        our_tickers = pd.read_csv(
            POLYGON_DATA_PATH + f"test/tickers/{START_DATE.date().isoformat()}.csv",
            index_col=0,
            keep_default_na=False,
            na_values=['#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
        )  # There is a stock named 'NA'. Avoid pandas treating it as N/A.
        our_tickers = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]]
        our_tickers = our_tickers[our_tickers["active"]]

        # Sometimes a cik can be empty. This prevents merge function from working.
        our_tickers['cik'] = our_tickers['cik'].replace('', np.nan)
        our_tickers['cik'] = our_tickers['cik'].astype(float)

        our_tickers.reset_index(inplace=True, drop=True)

        # Initialize tickers_all with Timestamp start_date
        our_tickers["start_date"] = START_DATE  # pd.Timestamp
        our_tickers["end_date"] = pd.NaT        # pandas datetime64[ns] compatible

    elif day > START_DATE and day <= END_DATE:
        # Get new ticker list to update ours
        tickers_day_i = pd.read_csv(
            POLYGON_DATA_PATH + f"test/tickers/{day.date().isoformat()}.csv",
            index_col=0,
            keep_default_na=False,
            na_values=['#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
        )
        tickers_day_i['cik'] = tickers_day_i['cik'].replace('', np.nan)
        tickers_day_i['cik'] = tickers_day_i['cik'].astype(float)

        tickers_day_i.sort_values(['last_updated_utc'], inplace=True)

        tickers_day_i = tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]]
        tickers_day_i = tickers_day_i[tickers_day_i["active"]]
        tickers_day_i.reset_index(inplace=True, drop=True)

        # Remove duplicates
        duplicated = tickers_day_i[tickers_day_i["ticker"].duplicated(keep=False)]
        indices_to_remove = duplicated["ticker"].duplicated(keep='last')
        tickers_day_i.drop(list(indices_to_remove[indices_to_remove].index), inplace=True)
        tickers_day_i.reset_index(drop=True, inplace=True)

        # Preliminary check: no duplicates
        if our_tickers.duplicated().any():
            raise Exception("There are duplicates in our_tickers!")
        if tickers_day_i.duplicated().any():
            raise Exception("There are duplicates in tickers_day_i!")

        # DELISTINGS
        indicator_delisted = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]].merge(
            tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]],
            on=["ticker", "name", "active", "cik", "composite_figi", "type"],
            how='left', indicator=True
        )

        indicator_delisted['_merge'] = np.where(
            our_tickers["active"],
            indicator_delisted['_merge'],
            "both"
        )  # ERROR FIX: already inactive stocks should not be delisted again

        indicator_delisted = indicator_delisted["_merge"]
        delisted_tickers = our_tickers[indicator_delisted == "left_only"]

        # NEW LISTINGS
        indicator_new = tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]].merge(
            our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]],
            on=["ticker", "name", "active", "cik", "composite_figi", "type"],
            how='left', indicator=True
        )
        indicator_new = indicator_new["_merge"]
        new_tickers = tickers_day_i[indicator_new == "left_only"]

        # PROCESS DELISTINGS
        # Get previous trading day (already a pd.Timestamp)
        idx = market_days.index(day)
        previous_day = market_days[idx - 1]  # pd.Timestamp

        # Assign end_date as Timestamp (compatible with datetime64[ns])
        our_tickers.loc[indicator_delisted == "left_only", "end_date"] = previous_day
        our_tickers.loc[indicator_delisted == "left_only", "active"] = False

        # PROCESS NEW LISTINGS
        our_tickers = pd.concat([our_tickers, new_tickers], ignore_index=True)
        # Fill missing start_date with current day (pd.Timestamp)
        our_tickers['start_date'] = our_tickers['start_date'].fillna(value=day)

        # Final checks
        if our_tickers[["ticker", "name", "active", "type", "start_date"]].isnull().values.any():
            raise Exception("There are missing values in required columns.")

        print(f'{day.date().isoformat()}: Amount of stocks {len(our_tickers)}')

        # Finalize if last day
        if day == END_DATE:
            our_tickers["end_date"] = our_tickers["end_date"].fillna(value=END_DATE)
            our_tickers = our_tickers.sort_values(by=["ticker", "end_date"]).reset_index(drop=True)
            our_tickers[["ticker", "name", "active", "start_date", "end_date", "type", "cik", "composite_figi"]] \
                .to_csv("../data/tickers_v1.csv")

2016-05-24 00:00:00
2023-05-24 00:00:00
2016-05-25: Amount of stocks 5759
2016-05-26: Amount of stocks 5769
2016-05-27: Amount of stocks 5775
2016-05-31: Amount of stocks 5784
2016-06-01: Amount of stocks 5788
2016-06-02: Amount of stocks 5798
2016-06-03: Amount of stocks 5804
2016-06-06: Amount of stocks 5806
2016-06-07: Amount of stocks 5808
2016-06-08: Amount of stocks 5809
2016-06-09: Amount of stocks 5818
2016-06-10: Amount of stocks 5826
2016-06-13: Amount of stocks 5835
2016-06-14: Amount of stocks 5842
2016-06-15: Amount of stocks 5848
2016-06-16: Amount of stocks 5854
2016-06-17: Amount of stocks 5858
2016-06-20: Amount of stocks 5864
2016-06-21: Amount of stocks 5874
2016-06-22: Amount of stocks 5883
2016-06-23: Amount of stocks 5890
2016-06-24: Amount of stocks 5892
2016-06-27: Amount of stocks 5898
2016-06-28: Amount of stocks 5910
2016-06-29: Amount of stocks 5913
2016-06-30: Amount of stocks 5917
2016-07-01: Amount of stocks 5924
2016-07-05: Amount of stocks 5933
2016-07-

We also create a function to retrieve the ticker list.

In [34]:
def get_tickers(v=5):
    """
    Retrieve the ticker list. Default is 5.
    """
    tickers = pd.read_csv(
        f"../data/tickers_v{v}.csv",
        parse_dates=["start_date", "end_date"],
        index_col=0,
        keep_default_na=False,
        na_values=["#N/A","#N/AN/A","#NA","-1.#IND","-1.#QNAN","-NaN","-nan","1.#IND","1.#QNAN","<NA>","N/A","NULL","NaN","None","n/a","nan","null",],
    )
    tickers["start_date"] = pd.to_datetime(tickers["start_date"]).dt.date
    tickers["end_date"] = pd.to_datetime(tickers["end_date"]).dt.date

    # This will only be applied in future notebooks.
    if tickers.columns.isin(["start_data", "end_data"]).any():
        tickers["start_data"] = pd.to_datetime(tickers["start_data"]).dt.date
        tickers["end_data"] = pd.to_datetime(tickers["end_data"]).dt.date

    return tickers

In [ ]:
tickers_v1 = get_tickers(1)

In [ ]:
tickers_v1[tickers_v1["ticker"] == "VFS"]

In [ ]:
tickers_v1[tickers_v1["ticker"] == "FB"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
10551,FB,FACEBOOK INC CL A COM STK (DE),False,2016-05-24,2016-10-24,,1326801.0,BBG000MM2P62
10552,FB,"Facebook, Inc. Class A",False,2016-10-25,2019-09-23,CS,1326801.0,BBG000MM2P62
10553,FB,"Facebook, Inc. Class A",False,2019-09-24,2019-09-24,CS,,BBG000MM2P62
10554,FB,"Facebook, Inc. Class A",False,2019-09-25,2021-10-29,CS,1326801.0,BBG000MM2P62
10555,FB,"Meta Platforms, Inc. Class A Common Stock",False,2021-11-01,2022-06-09,CS,1326801.0,BBG000MM2P62


In [36]:
tickers_v1[tickers_v1["ticker"] == "META"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
17928,META,"Meta Platforms, Inc. Class A Common Stock",True,2022-06-09,2023-05-24,CS,1326801.0,BBG000MM2P62


In [37]:
print(len(tickers_v1))

31316


# 3.4 Checks

1. Are SPACs handled correctly? We should expect that when they IPO a company, that they get delisted. Then one day after the delisting the new-born company should be listed. We will take a look at VFS. On 2023-8-15 it was IPO'd by the SPAC named BSAQ. So we should expect the delisting date of BSAQ to be 2023-8-14.


# Why is VFS missing now! 

In [38]:
tickers_v1[tickers_v1["ticker"] == "VFS"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi


In [39]:
tickers_v1[tickers_v1["ticker"] == "BSAQ"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
4340,BSAQ,Black Spade Acquisition Co,True,2021-09-07,2023-05-24,CS,1851908.0,


2. Let's check SVB which went bankrupt and HTZ which went from OTC to listed.

In [40]:
tickers_v1[tickers_v1["ticker"] == "SIVB"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
25542,SIVB,SVB FINANCIAL GROUP,False,2016-05-24,2016-10-24,,719739.0,BBG000BT0CM2
25543,SIVB,SVB Financial Group,False,2016-10-25,2019-09-23,CS,719739.0,BBG000BT0CM2
25544,SIVB,SVB Financial Group,False,2019-09-24,2019-09-24,CS,,BBG00G9F43F4
25545,SIVB,SVB Financial Group,False,2019-09-25,2023-03-27,CS,719739.0,BBG000BT0CM2


In [41]:
tickers_v1[tickers_v1["ticker"] == "HTZ"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
14079,HTZ,HERTZ GLOBAL HOLDINGS INC COM,False,2016-05-24,2016-07-01,CS,1364479.0,
14080,HTZ,"HERTZ GLOBAL HOLDINGS, INC. COM (DE)",False,2016-07-05,2016-10-24,CS,1657853.0,BBG00D5SHJH6
14081,HTZ,"Hertz Global Holdings, Inc.",False,2016-10-25,2020-10-29,CS,1657853.0,BBG00D5SHJH6
14082,HTZ,"Hertz Global Holdings, Inc Common Stock",True,2021-11-09,2023-05-24,CS,1657853.0,BBG011N57109


3. Sometimes tickers are re-used (e.g. META, but since it was an ETF it will not show up in our ticker list). Let's see if that has happened in our ticker list.

In [42]:
duplicated = tickers_v1[tickers_v1["ticker"].duplicated(keep=False)]
print(len(duplicated["ticker"].unique()))
print(len(duplicated))
duplicated.head()

7171
27383


,ticker,name,active,start_date,end_date,type,cik,composite_figi
0,A,"AGILENT TECHNOLOGIES, INC",False,2016-05-24,2016-10-24,CS,1090872.0,BBG000C2V3D6
1,A,Agilent Technologies,False,2016-10-25,2019-08-16,CS,1090872.0,BBG000C2V3D6
2,A,Agilent Technologies Inc.,False,2019-08-19,2019-09-23,CS,1090872.0,BBG000C2V3D6
3,A,Agilent Technologies Inc.,False,2019-09-24,2019-09-24,CS,,BBG000C2V3D6
4,A,Agilent Technologies Inc.,False,2019-09-25,2022-02-07,CS,1090872.0,BBG000C2V3D6


We will have some merging to do. However these are the "clean" ones. The next ones are just ridiculous and should not exist in the first place. 

In [43]:
from collections import Counter
print(Counter(duplicated["ticker"].values.tolist()).most_common(5))
duplicated[duplicated["ticker"] == "DGICA"].head(5)
# ???

[('CMS', 124), ('PRE', 117), ('EP', 113), ('CRESY', 24), ('DGICA', 20)]


,ticker,name,active,start_date,end_date,type,cik,composite_figi
8102,DGICA,Donegal Group Inc,False,2016-10-25,2019-09-23,CS,800457.0,BBG000JQJC22
8103,DGICA,Donegal Group Inc,False,2019-09-24,2019-09-24,CS,948046.0,BBG000JQJC22
8104,DGICA,Donegal Group Inc,False,2019-09-25,2022-08-25,CS,800457.0,BBG000JQJC22
8105,DGICA,Donegal Group Inc,False,2022-08-26,2022-08-26,CS,948046.0,BBG000JQJC22
8106,DGICA,Donegal Group Inc,False,2022-08-29,2022-09-02,CS,800457.0,BBG000JQJC22


In [44]:
duplicated[duplicated["ticker"] == "BORN"].head(5)

,ticker,name,active,start_date,end_date,type,cik,composite_figi
4096,BORN,CHINA NEW BORUN CORP ADR RPSGT ORD SHS,False,2016-05-24,2016-10-24,CS,1490366.0,BBG000QTHN26
4097,BORN,China New Borun Corporation,False,2016-10-25,2019-06-18,ADRC,1490366.0,BBG000QTHN26


On average there are 3 duplicates for duplicated tickers. When we take a look it seems that it happens a lot that the name/cik/composite_figi gets changed, even though it is the same company and ticker. For example for ZWS the name is "Zurn Water Solutions Corporation" on 2022-07-01 but on the next trading day (4th July was a stock holiday) the name changes to "Zurn Elkay Water Solutions Corporation". 

# 3.5 Merging duplicates
The most straightforward way to merge these duplicates is to see for every duplicate whether the the end_date (1st occurence) and start_date (2nd occurence) are consecutive *trading days*. Now you understand why we made the functions in notebook 2.

In [45]:
tickers_v1.head(3)

,ticker,name,active,start_date,end_date,type,cik,composite_figi
0,A,"AGILENT TECHNOLOGIES, INC",False,2016-05-24,2016-10-24,CS,1090872.0,BBG000C2V3D6
1,A,Agilent Technologies,False,2016-10-25,2019-08-16,CS,1090872.0,BBG000C2V3D6
2,A,Agilent Technologies Inc.,False,2019-08-19,2019-09-23,CS,1090872.0,BBG000C2V3D6


First we need to get the duplicates.

In [46]:
tickers_v1 = get_tickers(1)
duplicated = tickers_v1[tickers_v1["ticker"].duplicated(keep=False)]
duplicated.tail(3)

,ticker,name,active,start_date,end_date,type,cik,composite_figi
31313,ZYXI,ZYNEX INC,False,2019-02-12,2019-09-23,CS,846475.0,BBG000BJBXZ2
31314,ZYXI,ZYNEX INC,False,2019-09-24,2019-09-24,CS,,BBG000BJBXZ2
31315,ZYXI,ZYNEX INC,True,2019-09-25,2023-05-24,CS,846475.0,BBG000BJBXZ2


In [47]:
tickers_v1 = get_tickers(1)

In [48]:
tickers_v1 = tickers_v1.sort_values(by=['ticker', 'end_date'])
tickers_v1 = tickers_v1.reset_index(drop=True)
market_days = get_market_dates()

duplicated = tickers_v1[tickers_v1["ticker"].duplicated(keep=False)] # Get all duplicated, *including* the original

# Step 1: Get the indices of the rows that should be merged.
indices_duplicated = [] # looks like [['A', {1, 2, 3}], ['A', {4, 5}], ['B', {10, 11, 12, 13}]]
prev_index_and_row = None
prev_is_duplicate_and_back_to_back = False

for index, row in duplicated.iterrows():
    # Get attributes of previous ticker
    if prev_index_and_row is not None:
        prev_index = prev_index_and_row[0]
        prev_row = prev_index_and_row[1]
        prev_ticker = prev_row["ticker"]
        prev_name = prev_row["name"]
        prev_start_date = prev_row["start_date"]
        prev_end_date = prev_row["end_date"]
        prev_cik = prev_row["cik"]
        prev_figi = prev_row["composite_figi"]

    # Get attributes of current ticker
    current_index = index
    current_row = row
    current_ticker = current_row["ticker"]
    current_name = current_row["name"]
    current_start_date = current_row["start_date"]
    current_end_date = current_row["end_date"]
    current_cik = current_row["cik"]
    current_figi = current_row["composite_figi"]
    
    # Skip first index
    if prev_index_and_row is None:
        pass
    # Check if ticker duplicated and back-to-back
    elif prev_ticker == current_ticker and market_days[market_days.index(prev_end_date) + 1] == current_start_date:
        # If the previous was NOT duplicated/back-to-back, we need to create a new entry for the stock
        if prev_is_duplicate_and_back_to_back == False:
            indices_duplicated.append([current_ticker, {prev_index, current_index}])
        # Else the stock already exists in the list. Then simply append the indices.
        else:
            indices_duplicated[-1][-1].add(prev_index)
            indices_duplicated[-1][-1].add(current_index)
        
        # Update flag
        prev_is_duplicate_and_back_to_back = True
    else:
        prev_is_duplicate_and_back_to_back = False

    # Update prev_index_and_row for next iteration
    prev_index_and_row = (current_index, row)

In [49]:
print(len(indices_duplicated))
print(indices_duplicated[:3])

6804
[['A', {0, 1, 2, 3, 4, 5, 6}], ['AA', {7, 8, 9, 10, 11}], ['AAAP', {12, 13}]]


Very rarely, it happens that the same ticker, but not the same company, has duplicates. E.g. if in our ticker list the first 5 rows is the ticker AAA, but the start and end dates are (1, 2), (2, 3), (3, 4), (9, 10), (10, 11), this means that these are two different companies. Then indices_duplicated contains ['AAA', {1, 2, 3, 4}] and ['AAA, {10, 11}]. If it *is* the same company, it managed to get delisted to OTC and revive to get their listing back. However we stated that we are not interested in OTC so this is fine. Or something is wrong with Polygons data.

In [50]:
# See which tickers are duplicated in indices_duplicated.
tickers_already_checked = set() 
for ticker, indices in indices_duplicated:
    if ticker in tickers_already_checked:
        print(ticker)
    tickers_already_checked.add(ticker)

ADPT
AMTD
ASX
BBX
BGH
BV
CBL
CCC
CIVI
CLNY
COGT
COHR
CVU
DAVE
DD
DOW
EBR.B
EVER
GEC
GIG
GRAM
HCAC
HIL
HIVE
IGC
IMOS
JHB
LCA
LION
LOV
NAD
NE
NEA
NSR
OBE
OPP
PACD
PCI
RILYL
RILYZ
SCOR
SDRL
SE
SFR
SNCR
SPAQ
TBIO
TBIO
VAL
WF
ZTR


Now that we have a list of indices of the duplicated tickers, we can merge them together. We do this by looping over <code>indices_duplicated</code> and then changing all duplicated rows to get the correct values. Then we remove the duplicates.

In [51]:
# Step 2: Merge duplicated in tickers_all
"""
Which value is assigned:
    name: last
    active: last
    start_date: first
    end_date: last
    type: last (but does not matter as it is always CS or ADRC)
    cik: last value that is not NaN
    compositite_figi: last value that is not NaN
"""
for ticker, indices in indices_duplicated:
    # CAUTION: Make sure that indices is sorted! Else it can happen that end_date is before start_date. I only found this out later. Moral: Always do sanity checks.
    indices = sorted(list(indices))
    ticker_data_in_tickers_v1 = tickers_v1.iloc[indices, :]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("name")] = ticker_data_in_tickers_v1["name"].values[-1]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("active")] = ticker_data_in_tickers_v1["active"].values[-1]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("start_date")] = ticker_data_in_tickers_v1["start_date"].values[0]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("end_date")] = ticker_data_in_tickers_v1["end_date"].values[-1]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("type")] = ticker_data_in_tickers_v1["type"].values[-1]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("cik")] = ticker_data_in_tickers_v1["cik"].ffill().values[-1]
    tickers_v1.iloc[indices, tickers_v1.columns.get_loc("composite_figi")] = ticker_data_in_tickers_v1["composite_figi"].ffill().values[-1]

tickers_v1 = tickers_v1.drop_duplicates().reset_index(drop=True)

In [52]:
len(tickers_v1)

12463

Now only a fraction of the original duplicated tickers remain.

In [53]:
duplicated = tickers_v1[tickers_v1["ticker"].duplicated(keep=False)]
print(len(duplicated["ticker"].unique()))
pd.set_option('display.max_rows', None)

duplicated.head(5)

923


,ticker,name,active,start_date,end_date,type,cik,composite_figi
4,AAC,"AAC Holdings, Inc.",False,2016-05-24,2019-10-25,CS,1606180.0,BBG00K1Y3PT9
5,AAC,Ares Acquisition Corporation,True,2021-03-25,2023-05-24,CS,1829432.0,
69,ACAC,Acies Acquisition Corp. Class A Ordinary Share,False,2020-12-11,2021-06-21,CS,1823878.0,
70,ACAC,Acri Capital Acquisition Corporation Class A C...,True,2022-08-01,2023-05-24,CS,1914023.0,BBG0160DYSM3
85,ACCP,"Accelerated Pharma, Inc.",False,2016-12-09,2016-12-09,CS,,


We can see that some are the same company but not back-to-back. Some are different companies or went OTC and back, these are correct. However, for a lot of stocks, the first occurence only trades for a few days. That makes no sense. If you try to download data for these dates, you will see that there exists none.

# Ghost days (NOT AN ISSUE ANYMORE WITH POLYGON) (BTW this has been moved to later in this notebook)

So some stocks have a 'ghost' day just before their IPO. E.g. YGF was IPO'd on 2023-03-28. But on 2023-03-24 had a entry with start_date and end_date of just one day. This is the same with VCIG, which had 2 'ghost' days on 2023-03-22 and 2023-04-06. For SXTP, the ghost days were actually two. Investigating the stocks that only have 1 day in our ticker list also shows funds (that are NOT common stocks!).

Nevertheless, if start_date is equal to end_date, it's always unusable and something is wrong. So we will first remove all tickers that only exist for one day.

*This has been moved to after the second merging in 'more problems'. Else we will incorrectly remove the tickers that have one day. See the ticker CMCSA to see why.*

In [54]:
# ghost_days = tickers_v1[((tickers_v1["end_date"] - tickers_v1["start_date"]) == timedelta(days=0)) & 
#                         (tickers_v1["start_date"] <= market_days[market_days.index(END_DATE) - 1])]
# print(len(ghost_days))
# ghost_days.head(5)

In [55]:
# ###
# tickers_v1 = tickers_v1[((tickers_v1["end_date"] - tickers_v1["start_date"]) > timedelta(days=0)) | 
#                         (tickers_v1["start_date"] == market_days[market_days.index(END_DATE) - 1])]
# tickers_v1 = tickers_v1.reset_index(drop=True)
# len(tickers_v1)

The remaining duplicates are:

In [56]:
# duplicated = tickers_v1[tickers_v1["ticker"].duplicated(keep=False)]
# print(len(duplicated["ticker"].unique()))

# duplicated.head(5)

We need to keep in mind that the <code>start_date</code> and <code>end_date</code> may not be the start/end dates of the data. To determine the data dates, we need to loop through the ticker list and see whether the data exists. 

However, after we have downloaded our data, we can just infer it. So we will postpone this to avoid doing it twice.

In [57]:
len(tickers_v1)

12463

In [58]:
tickers_v1.to_csv("../data/tickers_v1_BEFORE_3.6.csv")

In [59]:
tickers_v1 = pd.read_csv("../data/tickers_v1_BEFORE_3.6.csv", index_col=0)

In [60]:
tickers_v1['name'] = tickers_v1['name'].astype(str)
tickers_v1['ticker'] = tickers_v1['ticker'].astype(str)
tickers_v1['start_date'] = pd.to_datetime(tickers_v1['start_date']).dt.date
tickers_v1['end_date'] = pd.to_datetime(tickers_v1['end_date']).dt.date

# 3.6 Removing incorrect classes
There are still some weird or incorrect stock classes that we have to remove. These were found by just looking through the ticker list.

These are:
- Funds. There are a LOT of them.
- Preferred stock/bonds
- Non-stocks, these have a uncapitalized letter in the ticker, such as "w" for warrants
- "Ex-distribution" or "When-issued" conditions

For the last 2 ones, it is understandable that they show up in ticker lists. However the first 2 should never.

Caution: we might remove some correct tickers. However that is better than having 100+ funds in the data... And it's also random so this is a less bad form of survivorship bias.

In [61]:
print(tickers_v1['name'].dtype)
print(tickers_v1['ticker'].dtype)

str
str


In [62]:
# Check for non-strings in each column you apply .lower() to
for col in ['name', 'ticker']:  # add any other columns you use
    non_strings = tickers_v1[col].apply(lambda x: not isinstance(x, str))
    if non_strings.any():
        print(f"Column '{col}' has non-string values at rows:")
        print(tickers_v1[non_strings][[col]])

# Keep in mind that "NAN" is a valid ticker, an actual stock. Just info - not actually relevant here since we are not deleting anything and are just figuring out data types. 


Column 'name' has non-string values at rows:
      name
922    NaN
923    NaN
924    NaN
925    NaN
3542   NaN
3620   NaN
3669   NaN
3783   NaN
5066   NaN
5138   NaN
5139   NaN
5852   NaN
7676   NaN
7678   NaN
8178   NaN
8179   NaN
8180   NaN
8181   NaN
9174   NaN
9483   NaN
10328  NaN
12441  NaN
Column 'ticker' has non-string values at rows:
     ticker
7734    NaN


This clearly shows the problem. Even after astype(str) in cell 54, you still have real NaN (float) values in your 'name' and 'ticker' columns. That's unusual because .astype(str) should convert every NaN into the string 'nan'.

In [63]:
funds = tickers_v1[tickers_v1['name'].apply(lambda s: 
        isinstance(s, str) and (
            "fund" in s.lower().split() \
            or "fund," in s.lower().split() \
            or "fnd" in s.lower().split() \
            or "fd" in s.lower().split() \
            or "aberdeen" in s.lower().split() \
            or "barings," in s.lower().split() \
            or "blackrock" in s.lower().split() \
            or "barclays" in s.lower().split() \
            or "bldrs" in s.lower().split() \
            or "contigent" in s.lower().split() \
            or "citigroup" in s.lower().split() \
            or "direxion" in s.lower().split() \
            or "mfs" in s.lower().split() \
            or "eaton" in s.lower().split() \
            or "calamos" in s.lower().split() \
            or "nuveen" in s.lower().split() \
            or "proshares" in s.lower().split() \
            or "suisse" in s.lower().split()\
            or "ishares" in s.lower().split()\
            or "jpmorgan" in s.lower().split()\
            or "invesco" in s.lower().split()\
            or "powershares" in s.lower().split()\
            or "gabelli" in s.lower().split() \
            or "morgan" in s.lower().split() \
            or "merrill" in s.lower().split() \
            or "merill" in s.lower().split() \
            or "merr" in s.lower().split() \
            or "etf" in s.lower().split() \
            or "etn" in s.lower().split() \
            or "etv" in s.lower().split() \
            or "index" in s.lower().split() \
            or "idx" in s.lower().split() \
            or "indx" in s.lower().split() \
            or "ctf" in s.lower().split() \
            or "pwrshrs" in s.lower().split() \
            or "pwrsh" in s.lower().split() \
            or "dbx" in s.lower().split() \
            or "msdw" in s.lower().split() \
            or "structured" in s.lower().split() \
            or "tr" in s.lower().split() \
            or "ubs" in s.lower().split() \
            or "bulletshares" in s.lower().split() \
            or "xai" in s.lower().split() \
            or "structrd" in s.lower().split() \
            or "structurd" in s.lower().split() \
            or "putnam" in s.lower().split() \
            or "citigrp" in s.lower().split() \
            or "citigp" in s.lower().split() \
            or "citicgroup" in s.lower().split() \
            or "mrgn" in s.lower().split() \
            or "lnk" in s.lower().split() \
            or "pines" in s.lower().split() \
            or "crt" in s.lower().split() \
            or "cert" in s.lower().split() \
            or "certificate" in s.lower().split() \
            or "lkd" in s.lower().split() \
            or "lknd" in s.lower().split() \
            or "velocityshs" in s.lower().split() \
            or "structred" in s.lower().split() \
            or "struct" in s.lower().split() \
            or "nt" in s.lower().split()\
            or "unit" in s.lower().split()\
            or "units" in s.lower().split()\
            or "\"quids\"" in s.lower().split() \
            or ("pioneer" in s.lower().split() and "trust" in s.lower().split())
        ))

        | tickers_v1['ticker'].isin(['CIP', 'FSMO', 'LOR', 'CMCA', 'EMD', 'HORI', 'KEMP', 'HCD', 'PMO', 'PCV', 'FSCO', 'AGB', 'FIV', 'GYB', 'FIV', 'IKBCO', 'MBINO', 'INDB.N', 'INDB', 'JBN', 'XFLT', 'VKA', 'VKC', 'VKS', 'TMT', 'TMB', 'RMT'])

        & ~tickers_v1['ticker'].isin(['JPM', 'EV', 'IVZ', 'IVR', 'BLK', 'TCPC', \
                'DJP', 'KMI', 'KMP', 'KMR', 'MHGC', 'MS', 'MWD', 'MR', 'MFUN', 'C', 'CS', 'CSR', 'UBS', 'CRO'])
        ]

print(len(funds))
funds.head(3)

1740


,ticker,name,active,start_date,end_date,type,cik,composite_figi
42,ABE,Aberdeen Emerging Markets Smaller Company Oppo...,False,2016-05-24,2017-08-04,CS,913662.0,NaN
125,ACP,AVENUE INCOME CR STRATEGIES FD COM SHS,False,2016-05-24,2016-10-24,CS,1503290.0,NaN
126,ACP,Aberdeen Income Crd Strategies,False,2019-04-02,2019-08-09,CS,1503290.0,BBG0017VSC04


In [64]:
tickers_pct = tickers_v1[tickers_v1['name'].str.contains('%')]
print(len(tickers_pct))
tickers_pct.head(3)

387


,ticker,name,active,start_date,end_date,type,cik,composite_figi
11,AAIN,Arlington Asset Investment Corp. 6.000% Senior...,False,2021-07-19,2022-08-19,NaN,1209028.0,NaN
99,ACGLN,"Arch Capital Group Ltd. Depositary Shares, eac...",True,2021-06-14,2023-05-24,CS,947484.0,NaN
219,AEB,AEGON N.V. 5.29964% Floating Rate Perpetual Ca...,False,2019-09-24,2019-09-24,NaN,769218.0,NaN


In [65]:
""" 
tickers_small_letter = tickers_v1[tickers_v1["ticker"].apply(lambda str: any(s.islower() for s in str))]
print(len(tickers_small_letter))
tickers_small_letter.head(3)
"""

# Remove things like warrants "ABCw".
# Check that the value is a string before checking for lowercase letters
tickers_small_letter = tickers_v1[
    tickers_v1["ticker"].apply(lambda x: isinstance(x, str) and any(c.islower() for c in x))
]
print(len(tickers_small_letter))
tickers_small_letter.head(3)


280


,ticker,name,active,start_date,end_date,type,cik,composite_figi
16,AANw,"The Aaron''s Company, Inc.",False,2020-11-25,2020-11-30,CS,1821393.0,BBG00WCNDCZ6
29,AAw,Alcoa Corporation,False,2016-10-18,2016-11-01,CS,1675149.0,BBG00B3T3HD3
78,ACAw,"Arcosa, Inc.",False,2018-10-16,2018-11-01,CS,1739445.0,BBG00JGMWFM9


In [66]:
tickers_preferred = tickers_v1[tickers_v1['name'].apply(lambda s: 
        isinstance(s, str) and (
               ("pf" in s.lower().split()) \
            or ("pfd" in s.lower().split()) \
            or ("pfr" in s.lower().split()) \
            or ("pref" in s.lower().split()) \
            or ("preferred" in s.lower().split()) \
            or ("exp" in s.lower().split()) \
            or ("due" in s.lower().split()) \
            or ("expiry" in s.lower().split())\
            or ("abs" in s.lower().split())\
            or ("warrant" in s.lower().split()) \
            or ("warrants" in s.lower().split()) \
            or ("crts" in s.lower().split())))
            ]
print(len(tickers_preferred))
tickers_preferred.head(3)

611


,ticker,name,active,start_date,end_date,type,cik,composite_figi
11,AAIN,Arlington Asset Investment Corp. 6.000% Senior...,False,2021-07-19,2022-08-19,NaN,1209028.0,NaN
98,ACGL.P,ARCH CAP GROUP LTD DS RPTG 1/1000 PFD SER E (BMU),False,2016-10-04,2016-10-24,NaN,NaN,NaN
99,ACGLN,"Arch Capital Group Ltd. Depositary Shares, eac...",True,2021-06-14,2023-05-24,CS,947484.0,NaN


In [67]:
when_issued_or_ex_distr = tickers_v1[tickers_v1['name'].apply(lambda s: 
        isinstance(s, str) and (
               ("when" in s.lower().split()) \
            or ("issued" in s.lower().split()) \
            or ("when-issued" in s.lower().split()) \
            or ("ex-distribution" in s.lower().split()) \
            or ("w.i." in s.lower().split()) \
            or ("wts" in s.lower().split())   ))  ]
print(len(when_issued_or_ex_distr))
when_issued_or_ex_distr.head(3)

136


,ticker,name,active,start_date,end_date,type,cik,composite_figi
166,ADEAV,Adeia Inc. Common Stock Ex-distribution When I...,False,2022-09-20,2022-10-03,CS,NaN,BBG019KN8702
312,AGFS.W,AGROFRESH SOLUTIONS INC WTS EXP 2/19/19 (DE),False,2016-05-24,2016-10-24,NaN,1592016.0,NaN
377,AIMCV,Altra Industrial Motion Corp. Common Stock Add...,False,2018-09-19,2018-10-01,CS,1374535.0,BBG000Q26KW0


In [68]:
import re

pattern = r"\.(?:WD|W|Z|V|U|P|PRTC|RTS|PRU|PRAC|PRDC|PRPC|PRBC|PREC)"
tickers_suffix = tickers_v1[tickers_v1["ticker"].str.contains(pattern, regex=True)]

print(len(tickers_suffix))
tickers_suffix.head(3)

188


,ticker,name,active,start_date,end_date,type,cik,composite_figi
44,ABEO.W,ABEONA THERAPEUTICS INC WT PUR COM,False,2016-05-24,2016-10-24,NaN,318306.0,NaN
98,ACGL.P,ARCH CAP GROUP LTD DS RPTG 1/1000 PFD SER E (BMU),False,2016-10-04,2016-10-24,NaN,NaN,NaN
214,ADXS.W,ADVAXIS INC WARRANTS EXP 07/15/2018,False,2016-05-24,2016-10-24,NaN,1100397.0,NaN


In [69]:
# https://www.nasdaqtrader.com/content/technicalsupport/specifications/dataproducts/nasdaqfifthcharactersuffixlist.pdf
tickers_units = tickers_v1[
    (tickers_v1["ticker"].str.replace(".", "").str.len() == 5)
    & (tickers_v1["ticker"].str[-1].isin(["G", "H", "I", "M", "N", "O", "P", "R", "T", "U", "V", "W", "Z"]))
]
print(len(tickers_units))
tickers_units.head(3)

412


,ticker,name,active,start_date,end_date,type,cik,composite_figi
44,ABEO.W,ABEONA THERAPEUTICS INC WT PUR COM,False,2016-05-24,2016-10-24,NaN,318306.0,NaN
98,ACGL.P,ARCH CAP GROUP LTD DS RPTG 1/1000 PFD SER E (BMU),False,2016-10-04,2016-10-24,NaN,NaN,NaN
99,ACGLN,"Arch Capital Group Ltd. Depositary Shares, eac...",True,2021-06-14,2023-05-24,CS,947484.0,NaN


In [70]:
indices_to_remove = set().union(*[set(funds.index), 
                                  set(tickers_pct.index),
                                  set(tickers_small_letter.index),
                                  set(tickers_preferred.index),
                                  set(when_issued_or_ex_distr.index),
                                  set(tickers_suffix.index),
                                  set(tickers_units.index),
                                  ])

print(len(indices_to_remove))

2661


In [71]:
print(len(tickers_v1))
tickers_v1 = tickers_v1.drop(index=indices_to_remove)
print(len(tickers_v1))

12463
9802


### Not Needed
Whether something is an ADR, however this has such a higher error rate that it is useless.

In [72]:
""" tickers_v1 = tickers_v1.reset_index(drop=True)
tickers_ADR = tickers_v1[(tickers_v1['name'].apply(lambda s: 
               ("n.v." in s.lower().split()) \
            or ("sa" in s.lower().split()) \
            or ("adr" in s.lower().split()) \
            or ("ads" in s.lower().split()) \
            or ("ltd" in s.lower().split()) \
            or ("plc" in s.lower().split()) \
            or ("depositary" in s.lower().split()) \
            or ("sponsored" in s.lower().split()) \
                )) & (tickers_v1['type'] == 'NONE')]

tickers_v1.iloc[tickers_ADR.index, 5] = 'ADRC'
tickers_v1.loc[tickers_v1['type'] == 'NONE', 'type'] = 'CS' """

' tickers_v1 = tickers_v1.reset_index(drop=True)\ntickers_ADR = tickers_v1[(tickers_v1[\'name\'].apply(lambda s: \n               ("n.v." in s.lower().split())             or ("sa" in s.lower().split())             or ("adr" in s.lower().split())             or ("ads" in s.lower().split())             or ("ltd" in s.lower().split())             or ("plc" in s.lower().split())             or ("depositary" in s.lower().split())             or ("sponsored" in s.lower().split())                 )) & (tickers_v1[\'type\'] == \'NONE\')]\n\ntickers_v1.iloc[tickers_ADR.index, 5] = \'ADRC\'\ntickers_v1.loc[tickers_v1[\'type\'] == \'NONE\', \'type\'] = \'CS\' '

# More problems
We have to fix the ticker inconsistencies. Sometimes when a ticker has multiple classes (e.g. BRK.B), the ticker list put a point. Sometimes not. We will fix this by cross-checking with the database. We see that the raw database always has a point with multiple classes. And never a point with some special conditions. It seems to follows the CQS conventions. However it's not 100%, you can see a ticker ZZO-A which has a dash... This is the only one. However the ticker list doesn't show this anyways.

However, tickers that are 5 characters long have no point for some reason.

### Not an issue anymore but might as well keep checking and cleaning any issues automatically!
#### ticker contains '/'

In [74]:
# Show all rows where ticker contains '/'
tickers_with_slash = tickers_v1[tickers_v1['ticker'].str.contains('/', na=False)]
print(len(tickers_with_slash))

0


In [75]:
# Change all / to a point
tickers_v1['ticker'] = tickers_v1['ticker'].str.replace('/', '.')

In [76]:
print(len(tickers_v1))
tickers_v1.head(3)

9802


,ticker,name,active,start_date,end_date,type,cik,composite_figi
0,A,Agilent Technologies Inc.,True,2016-05-24,2023-05-24,CS,1090872.0,BBG000C2V3D6
1,AA,Alcoa Corporation,True,2016-05-24,2023-05-24,CS,1675149.0,BBG00B3T3HD3
2,AAAP,Advanced Accelerator Applications S.A. America...,False,2016-05-24,2018-02-09,ADRC,1611787.0,BBG00K6FMBQ8


# BEWARE of running the code in the next cell! Of course, there's no m1 data. We are still cleaning the tickers list! And we also do NOT want to remove .A and .B tickers at all!

# Also, I don't think there's any need for this given we already remove the relevant .X suffices in a previous cell.

In [77]:
""" # Remove all points, unless its a .A or .B AND the database contains it 
files = os.listdir(POLYGON_DATA_PATH + "test/m1")
tickers_in_files = [file_name[:-8] for file_name in files] # Removes the last 8 characters from each filename like .parquet

tickers_v1["ticker"] = tickers_v1["ticker"].apply(
    lambda s: (
        s.replace(".", "")
        if not ((".A" in s or ".B" in s) and s in tickers_in_files)
        else s
    )
) """

' # Remove all points, unless its a .A or .B AND the database contains it \nfiles = os.listdir(POLYGON_DATA_PATH + "test/m1")\ntickers_in_files = [file_name[:-8] for file_name in files] # Removes the last 8 characters from each filename like .parquet\n\ntickers_v1["ticker"] = tickers_v1["ticker"].apply(\n    lambda s: (\n        s.replace(".", "")\n        if not ((".A" in s or ".B" in s) and s in tickers_in_files)\n        else s\n    )\n) '

In [78]:
""" # If a ticker is 5 long, remove the point if its .A or .B and the database contains it.
tickers_v1["ticker"] = tickers_v1["ticker"].apply(
    lambda s: (
        s.replace(".", "")
        if (len(s.replace(".", "")) == 5) and (".A" in s or ".B" in s) and s.replace(".", "") in tickers_in_files
        else s
    )
) """

' # If a ticker is 5 long, remove the point if its .A or .B and the database contains it.\ntickers_v1["ticker"] = tickers_v1["ticker"].apply(\n    lambda s: (\n        s.replace(".", "")\n        if (len(s.replace(".", "")) == 5) and (".A" in s or ".B" in s) and s.replace(".", "") in tickers_in_files\n        else s\n    )\n) '

In [79]:
# assert False

**Rerun the merging!** 

Some /A are corrected to .A which means that it should be merged with the corresponding .A ticker. Search in the ticker list for 'BRK' and you will see what I mean. Go back and rerun the cells for merging.

After merging again, we save the merged and cleaned ticker list to <code>tickers_v2.csv</code> and give an unique identifier.

In [ ]:
# Remove tickers that only have 1 day of history and are not just listed.

import bisect

# market_days sorted ascending, assume it contains datetime.date objects
print(type(END_DATE))
print(type(market_days[0]))  # check first element
print(END_DATE in market_days)  # should be True

# Convert END_DATE to datetime.date for comparison with market_days
if hasattr(END_DATE, 'date'):
    end_date_for_bisect = END_DATE.date()   # Timestamp -> date
else:
    end_date_for_bisect = END_DATE           # assume already date

print(END_DATE in market_days)  # should be True

# Now bisect works because both sides are date objects
pos = bisect.bisect_left(market_days, end_date_for_bisect)
if pos == 0:
    raise ValueError(f"No trading day before {END_DATE}")
prev_trading_day = market_days[pos - 1]      # datetime.date

# Convert prev_trading_day to Timestamp for comparison with start_date
mask = ((tickers_v1["end_date"] - tickers_v1["start_date"]) > timedelta(days=0)) | \
       (tickers_v1["start_date"] == pd.Timestamp(prev_trading_day))

# Validate
print(f"Total rows: {len(tickers_v1)}")
print(f"Rows to keep: {mask.sum()}")
print(f"Rows to remove: {(~mask).sum()}")

<class 'pandas.Timestamp'>
<class 'pandas.Timestamp'>
True
True


TypeError: Cannot compare Timestamp with datetime.date. Use ts == pd.Timestamp(date) or ts.date() == date instead.

In [81]:
# Apply filter
tickers_v1 = tickers_v1[mask].reset_index(drop=True)
print(f"\nNew length: {len(tickers_v1)}")


New length: 9629


In [82]:
tickers_v1["ID"] = tickers_v1["ticker"] + '-' + tickers_v1["start_date"].astype(str)

In [83]:
print(f"Total tickers: {len(tickers_v1)}")
print(f"Unique tickers: {len(tickers_v1['ticker'].unique())}")

tickers_v1 = tickers_v1.reset_index(drop=True)
tickers_v1 = tickers_v1[["ID", "ticker", "name", "active", "start_date", "end_date", "type", "cik", "composite_figi"]]
tickers_v1.to_csv("../data/tickers_v2.csv")

Total tickers: 9629
Unique tickers: 9297


In the beginning we had 49000 tickers. After removing the duplicates, incorrect classes and making sure the ticker conventions are correct, we have reduced this to ~~18291~~ 9625 tickers and 9293 unique tickers

The amount of tickers that have data is even smaller. As you will see, this is because a lot of tickers are 'ghost tickers' that have no data.

In [84]:
# assert False

We can see that some are the same company but not back-to-back. Some are different companies or went OTC and back, these are correct. However, for a lot of stocks, the first occurence only trades for a few days. That makes no sense. If you try to download data for these dates, you will see that there exists none.

# Ghost days (NOT AN ISSUE ANYMORE WITH POLYGON)

So some stocks have a 'ghost' day just before their IPO. E.g. YGF was IPO'd on 2023-03-28. But on 2023-03-24 had a entry with start_date and end_date of just one day. This is the same with VCIG, which had 2 'ghost' days on 2023-03-22 and 2023-04-06. For SXTP, the ghost days were actually two. Investigating the stocks that only have 1 day in our ticker list also shows funds (that are NOT common stocks!).

Nevertheless, if start_date is equal to end_date, it's always unusable and something is wrong. So we will first remove all tickers that only exist for one day.

*This has been moved to after the second merging in 'more problems'. Else we will incorrectly remove the tickers that have one day. See the ticker CMCSA to see why.*

In [107]:
tickers_v1.head(3)

,ID,ticker,name,active,start_date,end_date,type,cik,composite_figi
0,A-2016-05-24,A,Agilent Technologies Inc.,True,2016-05-24,2023-05-24,CS,1090872.0,BBG000C2V3D6
1,AA-2016-05-24,AA,Alcoa Corporation,True,2016-05-24,2023-05-24,CS,1675149.0,BBG00B3T3HD3
2,AAAP-2016-05-24,AAAP,Advanced Accelerator Applications S.A. America...,False,2016-05-24,2018-02-09,ADRC,1611787.0,BBG00K6FMBQ8


In [127]:
# --- CONVERT DATE RANGE TO PANDAS TIMESTAMP ---
# Changed: Use pd.Timestamp instead of date to avoid dtype issues
START_DATE = pd.Timestamp(date(2016, 5, 24))
END_DATE = pd.Timestamp(date(2023, 5, 24))

# Assumed existing functions (they must return date-like objects)
# We'll convert their results to Timestamp as well
market_dates_raw = get_market_dates()  # list of datetime.date or Timestamp
market_days = [pd.Timestamp(d) for d in market_dates_raw]  # force Timestamp

# Adjust to trading days – convert to date for comparison, then back to Timestamp
START_DATE = pd.Timestamp(first_trading_date_after_equal(START_DATE.date()))
END_DATE = pd.Timestamp(last_trading_date_before_equal(END_DATE.date()))

print(START_DATE)
print(END_DATE)
print(market_days[:5], market_days[-5:])
print(market_days[market_days.index(END_DATE) - 1])

tickers_v1["start_date"] = pd.to_datetime(tickers_v1["start_date"])
tickers_v1["end_date"] = pd.to_datetime(tickers_v1["end_date"])

ghost_days = tickers_v1[((tickers_v1["end_date"] - tickers_v1["start_date"]) == timedelta(days=0)) & 
                        (tickers_v1["start_date"] <= market_days[market_days.index(END_DATE) - 1])]
print(len(ghost_days))
ghost_days.head(5)

# ghost_days_test with 600 days or around 3 years
# ghost_days_test = tickers_v1[((tickers_v1["end_date"] - tickers_v1["start_date"]) == timedelta(days=600)) & 
#                         (tickers_v1["start_date"] <= market_days[market_days.index(END_DATE) - 1])]
# print(len(ghost_days_test))
# ghost_days_test.head(5)


2016-05-24 00:00:00
2023-05-24 00:00:00
[Timestamp('2016-05-23 00:00:00'), Timestamp('2016-05-24 00:00:00'), Timestamp('2016-05-25 00:00:00'), Timestamp('2016-05-26 00:00:00'), Timestamp('2016-05-27 00:00:00')] [Timestamp('2024-04-15 00:00:00'), Timestamp('2024-04-16 00:00:00'), Timestamp('2024-04-17 00:00:00'), Timestamp('2024-04-18 00:00:00'), Timestamp('2024-04-19 00:00:00')]
2023-05-23 00:00:00
0


,ID,ticker,name,active,start_date,end_date,type,cik,composite_figi


In [ ]:
# NOT NEEDED ANYMORE COS GHOST DAYS ARE NOT AN ISSUE WITH POLYGON DATA ANYMORE.
# tickers_v1 = tickers_v1[((tickers_v1["end_date"] - tickers_v1["start_date"]) > timedelta(days=0)) | 
#                         (tickers_v1["start_date"] == market_days[market_days.index(END_DATE) - 1])]
# tickers_v1 = tickers_v1.reset_index(drop=True)
# len(tickers_v1)

The remaining duplicates are:

In [ ]:
# duplicated = tickers_v1[tickers_v1["ticker"].duplicated(keep=False)]
# print(len(duplicated["ticker"].unique()))

# duplicated.head(5)

We need to keep in mind that the <code>start_date</code> and <code>end_date</code> may not be the start/end dates of the data. To determine the data dates, we need to loop through the ticker list and see whether the data exists. 

However, after we have downloaded our data, we can just infer it. So we will postpone this to avoid doing it twice.

# 3.7 Updates
1. Run the first cell below to create a backup of <code>tickers_v3</code>. Later when we download data, we will compare the old to the new ticker list to determine which stocks and dates to update, instead of downloading everything.
1. Update END_DATE.
2. 3.1: Run the 3 cells with ###. This updates the folder of ticker lists.
3. Run the below cells to get the new <code>tickers_v1</code>.
4. 3.5, 3.6: Run everything (merge, remove incorrect classes) to get <code>tickers_v2</code>.

In [ ]:
tickers_v3 = get_tickers(3)
tickers_v3.to_csv("../data/tickers_v3_old.csv")

This is the tickers loop, however the first ticker list is tickers_v1 instead of the ticker list of the first day. This is much faster than redoing everything.

In [ ]:
from tickers import get_tickers
market_days = get_market_dates()

tickers_v1 = get_tickers(1)
CURRENT_END_DATE = tickers_v1['end_date'].max()
END_DATE = last_trading_date_before_equal(END_DATE)

market_days = get_market_dates()

for day in market_days:
    if day == CURRENT_END_DATE:
        # At the start, our ticker list is our current tickers_v1
        our_tickers = tickers_v1
        our_tickers.loc[our_tickers['active'], 'end_date'] = np.NaN
        our_tickers['cik'] = our_tickers['cik'].replace('', np.nan)
        our_tickers['cik'] = our_tickers['cik'].astype(float)

    elif day > CURRENT_END_DATE and day <= END_DATE:
        # Get new ticker list to update ours
        tickers_day_i = pd.read_csv(
            POLYGON_DATA_PATH + f"test/tickers/{day.isoformat()}.csv",
            index_col=0,
            keep_default_na=False,
            na_values=['#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', \
            '1.#QNAN', '<NA>', 'N/A', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
        )
        # Sometimes a cik can be empty. This prevents the merge function from working.
        tickers_day_i['cik'] = tickers_day_i['cik'].replace('', np.nan)
        tickers_day_i['cik'] = tickers_day_i['cik'].astype(float)

        tickers_day_i.sort_values(['last_updated_utc'], inplace=True)

        tickers_day_i = tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]]
        tickers_day_i = tickers_day_i[tickers_day_i["active"]]
        tickers_day_i.reset_index(inplace=True, drop=True)

        # Remove duplicates (very rare, but sometimes this happens due to duplicates in polygons ticker lists)
        duplicated = tickers_day_i[tickers_day_i["ticker"].duplicated(keep=False)]
        indices_to_remove = duplicated["ticker"].duplicated(keep='last')
        tickers_day_i.drop(list(indices_to_remove[indices_to_remove].index), inplace=True)
        tickers_day_i.reset_index(drop=True, inplace=True)

        # Preliminary check: no duplicates
        if our_tickers.duplicated().any():
            raise Exception("There are duplicates!")

        if tickers_day_i.duplicated().any():
            raise Exception("There are duplicates!")

        # DELISTINGS
        indicator_delisted = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]].\
            merge(tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]], \
                on=["ticker", "name", "active", "cik", "composite_figi", "type"], how='left', indicator=True)

        indicator_delisted['_merge'] = np.where(our_tickers["active"], indicator_delisted['_merge'], "both") # ERROR FIX: If in our ticker list we have already set it inactive, it should not be added to the list of delisted stocks again. By setting _merge to "both" we skip the already inactive stocks.

        indicator_delisted = indicator_delisted["_merge"] # Only get the indicator
        delisted_tickers = our_tickers[indicator_delisted == "left_only"]

        # NEW LISTINGS
        indicator_new = tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]].\
            merge(our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]], \
                on=["ticker", "name", "active", "cik", "composite_figi", "type"], 
                        how='left', indicator=True)
        indicator_new = indicator_new["_merge"]
        new_tickers = tickers_day_i[indicator_new == "left_only"]

        # PROCESS DELISTINGS
        previous_day = market_days[market_days.index(day) - 1] # Getting previous trading day
        our_tickers.loc[indicator_delisted == "left_only", "end_date"] = previous_day
        our_tickers.loc[indicator_delisted == "left_only", "active"] = False
        
        # PROCESS NEW LISTINGS
        our_tickers = pd.concat([our_tickers, new_tickers])
        our_tickers.reset_index(inplace=True, drop=True)
        our_tickers['start_date'].fillna(value=day, inplace=True)

        # Final checks
        if our_tickers[["ticker", "name", "active", "type", "start_date"]].isnull().values.any():
            #null_data = our_tickers[our_tickers[["ticker", "name", "active", "type", "start_date"]].isnull().any(axis=1)]
            raise Exception("There are missing values.")
        
        print(f'{day.isoformat()}: Amount of stocks {len(our_tickers)}')
        
        # Finalize
        if day == END_DATE:
            our_tickers["end_date"].fillna(value=END_DATE, inplace=True)
            our_tickers = our_tickers.sort_values(by=["ticker", "end_date"]).reset_index(drop=True)
            our_tickers[["ticker", "name", "active", "start_date", "end_date", "type", "cik", "composite_figi"]].\
                to_csv("../data/tickers_v1.csv")